In [45]:
import torch

In [46]:
# Create a 2D tensor with shape (5, 2)
# Values: 0, 1, 2, ..., 9 arranged in 5 rows and 2 columns
t = torch.arange(0, 10).reshape(5, 2)
print(t)

tensor([[0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
        [8, 9]])


In [47]:
# Print the stride of the tensor
# Stride (2, 1) means:
#   - Move 2 elements in memory to go to the next row
#   - Move 1 element in memory to go to the next column
print(t.stride())

(2, 1)


### Understanding `tensor.stride()`
- A stride tells PyTorch how many elements (in memory) it must skip to move to the next item along a dimension.
- For the 2D tensor above (shape `5 x 2`), the first stride value corresponds to moving one row down, and the second value corresponds to moving one column across.
- Strides depend on how the tensor is laid out in memory; reshaping or transposing changes them even if the data visually looks similar.
- Contiguous tensors typically have decreasing strides (row-major order), but operations like `transpose` can swap stride values without copying data.

In [48]:
# Compare strides across different tensor operations

# 1. Original tensor: standard row-major layout
print("Original tensor:")
print(t)
print("Shape:", t.shape)  # (5, 2)
print("Strides:", t.stride())  # (2, 1) - move 2 for next row, 1 for next column

# 2. Transpose: swaps dimensions and strides without copying data
print("\nAfter transpose (swap rows/cols):")
t_T = t.t()  # Transpose operation
print(t_T)
print("Shape:", t_T.shape)  # (2, 5) - dimensions swapped
print("Strides:", t_T.stride())  # (1, 2) - strides swapped too!

# 3. Flatten to 1D: contiguous view with single stride
print("\nView as 1D without copying:")
t_flat = t.view(-1)  # Reshape to 1D
print(t_flat)
print("Shape:", t_flat.shape)  # (10,)
print("Strides:", t_flat.stride())  # (1,) - move 1 element for next item


# 4. Slicing with step: creates non-contiguous view
print("\nNon-contiguous slice (every other row):")
t_slice = t[::2]  # Take every 2nd row
print(t_slice)
print("Shape:", t_slice.shape)  # (3, 2) - 3 rows, 2 columns
print("Strides:", t_slice.stride())  # (4, 1) - skip 4 elements for next row!

Original tensor:
tensor([[0, 1],
        [2, 3],
        [4, 5],
        [6, 7],
        [8, 9]])
Shape: torch.Size([5, 2])
Strides: (2, 1)

After transpose (swap rows/cols):
tensor([[0, 2, 4, 6, 8],
        [1, 3, 5, 7, 9]])
Shape: torch.Size([2, 5])
Strides: (1, 2)

View as 1D without copying:
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
Shape: torch.Size([10])
Strides: (1,)

Non-contiguous slice (every other row):
tensor([[0, 1],
        [4, 5],
        [8, 9]])
Shape: torch.Size([3, 2])
Strides: (4, 1)


In [49]:
# Check contiguity and demonstrate how permute affects it
print("Original tensor contiguous:", t.is_contiguous())
print("Original strides:", t.stride())

print("Contiguous:", t.is_contiguous())

# Permute dimensions (swap dimensions 0 and 1)print("Strides:", t_permuted.stride())

t_permuted = t.permute(1, 0)  # Need to specify all dimensionsprint("Shape:", t_permuted.shape)
print("\nAfter permute(1,0):")
print("not contiguous: ", t_permuted.is_contiguous())
# make t_permuted contiguous
t_new = t_permuted.contiguous()
print("should be contiguous: ", t_new.is_contiguous())

Original tensor contiguous: True
Original strides: (2, 1)
Contiguous: True

After permute(1,0):
not contiguous:  False
should be contiguous:  True


In [50]:
# tile swapping example
import numpy as np


a = np.arange(16).reshape(4, 4)

a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15]])

In [51]:
# Demonstrate tile swapping using vstack
# vstack takes a sequence (tuple/list) of arrays to stack vertically
v1 = np.vstack((a[0:2, 0:2], a[0:2, 2:4]))
print("Top-left and top-right tiles stacked:")
print(v1)

# Alternative: stack bottom tiles
v2 = np.vstack((a[2:4, 0:2], a[2:4, 2:4]))
print("\nBottom-left and bottom-right tiles stacked:")
print(v2)

h = np.hstack((v1, v2))
print(h)

Top-left and top-right tiles stacked:
[[0 1]
 [4 5]
 [2 3]
 [6 7]]

Bottom-left and bottom-right tiles stacked:
[[ 8  9]
 [12 13]
 [10 11]
 [14 15]]
[[ 0  1  8  9]
 [ 4  5 12 13]
 [ 2  3 10 11]
 [ 6  7 14 15]]


In [52]:
from numpy.lib.stride_tricks import as_strided

# Zero-copy tile swapping using views and advanced indexing
print("Original 4x4 array:")
print(a)
print("Shape:", a.shape)

# Method 1: Using views to swap tiles without copying data
print("\n=== Zero-Copy Tile Swapping ===")

# Create a view that rearranges the tiles
# We want to swap: top-left ↔ bottom-right, top-right ↔ bottom-left
# This uses fancy indexing to create a new view of the same data

# Define tile indices for a 4x4 array split into 2x2 tiles
tile_indices = np.array([[2, 3, 0, 1],  # Row mapping: bottom rows first, then top
                        [2, 3, 0, 1],   # Same for second row
                        [0, 1, 2, 3],   # Top rows go to bottom
                        [0, 1, 2, 3]])  # Same pattern

# Create column indices that swap left↔right within each 2x2 block
col_indices = np.array([2, 3, 0, 1])  # Columns: right half first, then left half

# Apply the zero-copy rearrangement
swapped_view = a[np.ix_(tile_indices.diagonal(), col_indices)]
print("\nTile-swapped view (zero copy):")
print(swapped_view)

# Method 2: More explicit tile swapping using slicing and concatenation
print("\n=== Alternative: Block-wise swapping ===")

# Extract tiles as views (no copying)
top_left = a[0:2, 0:2]     # Original position: top-left
top_right = a[0:2, 2:4]    # Original position: top-right  
bottom_left = a[2:4, 0:2]  # Original position: bottom-left
bottom_right = a[2:4, 2:4] # Original position: bottom-right

print("Original tiles:")
print(f"Top-left:\n{top_left}")
print(f"Top-right:\n{top_right}")
print(f"Bottom-left:\n{bottom_left}")
print(f"Bottom-right:\n{bottom_right}")

# Note: vstack and hstack create copies, but we can show the concept
# For true zero-copy, you'd need to work with strided views or use
# libraries like numpy.lib.stride_tricks

print("\n=== Using stride_tricks for true zero-copy ===")

# Create a view that reorders the data without copying
# This is advanced but demonstrates true zero-copy operations
original_strides = a.strides
print(f"Original strides: {original_strides}")

# Create a custom strided view that effectively swaps the tiles
# This requires careful calculation of new strides and shapes
swapped_strides = (original_strides[0] * 2, original_strides[1] * 2, 
                   original_strides[0], original_strides[1])
swapped_shape = (2, 2, 2, 2)  # 2x2 tiles, each 2x2 in size

# View the array as 2x2 blocks
tiled_view = as_strided(a, shape=swapped_shape, strides=swapped_strides)
print(f"\nTiled view shape: {tiled_view.shape}")
print("Tiled view (each 2x2 block):")
for i in range(2):
    for j in range(2):
        print(f"Tile [{i},{j}]:")
        print(tiled_view[i, j])
        print()

print("Memory usage comparison:")
print(f"Original array: {a.nbytes} bytes")
print("Views use same memory - no additional allocation!")
print(f"All views share data: {np.shares_memory(a, top_left)}")

Original 4x4 array:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
Shape: (4, 4)

=== Zero-Copy Tile Swapping ===

Tile-swapped view (zero copy):
[[10 11  8  9]
 [14 15 12 13]
 [10 11  8  9]
 [14 15 12 13]]

=== Alternative: Block-wise swapping ===
Original tiles:
Top-left:
[[0 1]
 [4 5]]
Top-right:
[[2 3]
 [6 7]]
Bottom-left:
[[ 8  9]
 [12 13]]
Bottom-right:
[[10 11]
 [14 15]]

=== Using stride_tricks for true zero-copy ===
Original strides: (32, 8)

Tiled view shape: (2, 2, 2, 2)
Tiled view (each 2x2 block):
Tile [0,0]:
[[0 1]
 [4 5]]

Tile [0,1]:
[[2 3]
 [6 7]]

Tile [1,0]:
[[ 8  9]
 [12 13]]

Tile [1,1]:
[[10 11]
 [14 15]]

Memory usage comparison:
Original array: 128 bytes
Views use same memory - no additional allocation!
All views share data: True


In [53]:
# Check if NumPy array is contiguous (note: different method than PyTorch)

print("NumPy array contiguous:", a.flags['C_CONTIGUOUS'])print("Every-other strides:", every_other.strides)

print("NumPy array strides:", a.strides)print("\nEvery-other slice contiguous:", every_other.flags['C_CONTIGUOUS'])

every_other = a[::2, ::2]

# For comparison, check some views# Non-contiguous slice

top_left = a[0:2, 0:2]

print("\nTop-left slice contiguous:", top_left.flags['C_CONTIGUOUS'])print("Top-left strides:", top_left.strides)

SyntaxError: invalid syntax (2776989120.py, line 3)